# Practice: overfitting, regularization and leakage
*AI from scratch, chapter 11* · lesson: https://ihor.sh/ai/11-overfitting.html

The everyday tools against overfitting, in scikit-learn: honest splits, a validation curve, ridge tuned by
cross-validation, learning curves, and the most common way people fool themselves: **data leakage**.
Fill in each `...` under `# YOUR CODE HERE`, then run the check under it (✓ = right, ✗ = not yet).

In [ ]:
# Helpers: run this cell first. check() marks an answer ✓ or ✗; stuck? hint(n) gives the next hint for
# exercise n, answer(n) shows a solution.
import base64 as _b64
_HINTS = {1: ['Split X, y into 60% train and 40% temp with test_size=0.4, random_state=0.', 'Split temp into 50% validation and 50% test with test_size=0.5, random_state=0 (each getting 20% of original).'], 2: ['Initialize train_err = [] and val_err = [].', 'For each d in degrees: Fit make_pipeline(PolynomialFeatures(d), LinearRegression()) on Xtr, ytr. Record MSE on Xtr in train_err and MSE on Xva in val_err.', 'Find best_d = list(degrees)[np.argmin(val_err)].'], 3: ['For each alpha, compute 5-fold cross-validation MSE on Xtr, ytr: -cross_val_score(..., scoring="neg_mean_squared_error").mean().', 'Identify best_alpha as the alpha yielding minimum cv_err.'], 4: ['Call learning_curve with make_pipeline(PolynomialFeatures(15), StandardScaler(), Ridge(alpha=1e-6)).', 'Use train_sizes=np.linspace(0.1, 1, 8), cv=5, scoring="neg_mean_squared_error".', 'Compute gap_small = va_err[0] - tr_err[0] and gap_big = va_err[-1] - tr_err[-1].'], 5: ['Leaky baseline: SelectKBest on full Xn, yn first, then cross_val_score on the 20 selected features.', 'Honest baseline: Put SelectKBest inside make_pipeline(SelectKBest(...), LogisticRegression(...)) before CV.']}
_ANSWERS = {1: 'WHRyLCBYcmVzdCwgeXRyLCB5cmVzdCA9IHRyYWluX3Rlc3Rfc3BsaXQoWCwgeSwgdGVzdF9zaXplPTAuNCwgcmFuZG9tX3N0YXRlPTApClh2YSwgWHRlLCB5dmEsIHl0ZSA9IHRyYWluX3Rlc3Rfc3BsaXQoWHJlc3QsIHlyZXN0LCB0ZXN0X3NpemU9MC41LCByYW5kb21fc3RhdGU9MCk=', 2: 'dHJhaW5fZXJyLCB2YWxfZXJyID0gW10sIFtdCmZvciBkIGluIGRlZ3JlZXM6CiAgICBtID0gbWFrZV9waXBlbGluZShQb2x5bm9taWFsRmVhdHVyZXMoZCksIExpbmVhclJlZ3Jlc3Npb24oKSkuZml0KFh0ciwgeXRyKQogICAgdHJhaW5fZXJyLmFwcGVuZChucC5tZWFuKChtLnByZWRpY3QoWHRyKSAtIHl0cikgKiogMikpOyB2YWxfZXJyLmFwcGVuZChucC5tZWFuKChtLnByZWRpY3QoWHZhKSAtIHl2YSkgKiogMikpCmJlc3RfZCA9IGxpc3QoZGVncmVlcylbaW50KG5wLmFyZ21pbih2YWxfZXJyKSldCnBsdC5wbG90KGRlZ3JlZXMsIHRyYWluX2VyciwgIm8tIiwgbGFiZWw9InRyYWluIik7IHBsdC5wbG90KGRlZ3JlZXMsIHZhbF9lcnIsICJvLSIsIGxhYmVsPSJ2YWxpZGF0aW9uIikKcGx0LnlzY2FsZSgibG9nIik7IHBsdC54bGFiZWwoImRlZ3JlZSIpOyBwbHQubGVnZW5kKCk7IHBsdC5zaG93KCk=', 3: 'Y3ZfZXJyID0gWy1jcm9zc192YWxfc2NvcmUobWFrZV9waXBlbGluZShQb2x5bm9taWFsRmVhdHVyZXMoMTUpLCBTdGFuZGFyZFNjYWxlcigpLCBSaWRnZShhbHBoYT1hKSksIFh0ciwgeXRyLCBjdj01LAogICAgICAgICAgICAgICAgICAgICAgICAgICBzY29yaW5nPSJuZWdfbWVhbl9zcXVhcmVkX2Vycm9yIikubWVhbigpIGZvciBhIGluIGFscGhhc10KYmVzdF9hbHBoYSA9IGFscGhhc1tpbnQobnAuYXJnbWluKGN2X2VycikpXQ==', 4: 'c2l6ZXMsIHRyLCB2YSA9IGxlYXJuaW5nX2N1cnZlKG1ha2VfcGlwZWxpbmUoUG9seW5vbWlhbEZlYXR1cmVzKDE1KSwgU3RhbmRhcmRTY2FsZXIoKSwgUmlkZ2UoYWxwaGE9MWUtNikpLCBYLCB5LCBjdj01LAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgdHJhaW5fc2l6ZXM9bnAubGluc3BhY2UoMC4xLCAxLCA4KSwgc2NvcmluZz0ibmVnX21lYW5fc3F1YXJlZF9lcnJvciIpCnRyX2VyciwgdmFfZXJyID0gLXRyLm1lYW4oMSksIC12YS5tZWFuKDEpCnBsdC5wbG90KHNpemVzLCB0cl9lcnIsICJvLSIsIGxhYmVsPSJ0cmFpbiIpOyBwbHQucGxvdChzaXplcywgdmFfZXJyLCAiby0iLCBsYWJlbD0idmFsaWRhdGlvbiIpCnBsdC55c2NhbGUoImxvZyIpOyBwbHQueGxhYmVsKCJ0cmFpbmluZyBleGFtcGxlcyIpOyBwbHQubGVnZW5kKCk7IHBsdC5zaG93KCkKZ2FwX3NtYWxsLCBnYXBfYmlnID0gdmFfZXJyWzBdIC0gdHJfZXJyWzBdLCB2YV9lcnJbLTFdIC0gdHJfZXJyWy0xXQ==', 5: 'WF9zZWwgPSBTZWxlY3RLQmVzdChmX2NsYXNzaWYsIGs9MjApLmZpdF90cmFuc2Zvcm0oWG4sIHluKQpsZWFreSA9IGNyb3NzX3ZhbF9zY29yZShMb2dpc3RpY1JlZ3Jlc3Npb24obWF4X2l0ZXI9MTAwMCksIFhfc2VsLCB5biwgY3Y9NSkubWVhbigpCmhvbmVzdCA9IGNyb3NzX3ZhbF9zY29yZShtYWtlX3BpcGVsaW5lKFNlbGVjdEtCZXN0KGZfY2xhc3NpZiwgaz0yMCksIExvZ2lzdGljUmVncmVzc2lvbihtYXhfaXRlcj0xMDAwKSksIFhuLCB5biwgY3Y9NSkubWVhbigp'}
_given = {}

def check(n, test, msg=""):
    try:
        ok = bool(test())
    except NameError as e:
        return print("✗ %s: run the exercise cell above first." % e)
    except Exception as e:
        if "NoneType" in str(e) or "ellipsis" in str(e):
            return print("✗ Not answered yet: replace the ... in exercise %d and run that cell. Stuck? hint(%d)" % (n, n))
        ok, msg = False, msg + " (the check stopped with %s: %s)" % (type(e).__name__, e)
    print("✓ Right." if ok else "✗ Not yet. " + msg + "\n  Stuck? hint(%d) for a hint, answer(%d) for a solution." % (n, n))

def hint(n):
    hints, i = _HINTS.get(n, []), _given.get(n, 0)
    if not hints:
        return print("No hints for this one: answer(%d) shows a solution." % n)
    print("Hint %d of %d: %s" % (min(i, len(hints) - 1) + 1, len(hints), hints[min(i, len(hints) - 1)]))
    _given[n] = i + 1

def answer(n):
    print(_b64.b64decode(_ANSWERS[n]).decode())
    print("\n(Put this in place of the ..., run it, then close it and try writing it yourself.)")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, learning_curve
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression

rng = np.random.default_rng(0)

## 1. Train, validation, test
300 noisy points from a hidden curve. Split them 60/20/20 into train, validation and test with two calls to
`train_test_split` (random_state=0 both times). Set `Xtr, Xva, Xte, ytr, yva, yte`.

In [ ]:
x = rng.uniform(-1, 1, 300); y = 0.8 * np.sin(2.5 * x) + 0.3 * x + 0.25 * rng.standard_normal(300)
X = x[:, None]
Xtr = Xva = Xte = ytr = yva = yte = None
# YOUR CODE HERE (stuck? hint(1), then answer(1))
...
print([len(a) for a in (Xtr, Xva, Xte)] if Xtr is not None else None)

In [ ]:
check(1, lambda: [len(a) for a in (Xtr, Xva, Xte)] == [180, 60, 60], 
      "Dataset of 300 points should split into 180 train (60%), 60 validation (20%), and 60 test (20%).")

## 2. The validation curve
For degrees 1–15, fit `make_pipeline(PolynomialFeatures(d), LinearRegression())` on train and record the mean
squared error on train and on validation. Set `train_err`, `val_err` (lists) and `best_d`: the degree with the
lowest validation error. Plot both curves.

In [ ]:
degrees = range(1, 16)
train_err = val_err = best_d = None
# YOUR CODE HERE (stuck? hint(2), then answer(2))
...
print("best degree:", best_d)

In [ ]:
check(2, lambda: all(a >= b - 1e-12 for a, b in zip(train_err, train_err[1:])), 
      "Training error should monotonically decrease as polynomial capacity increases.")
check(2, lambda: 3 <= best_d <= 9, 
      "Validation error should be minimized at a moderate polynomial degree (between 3 and 9).")

## 3. Ridge, tuned by cross-validation
A degree-15 polynomial overfits; ridge (L2) reins it in. For each `alpha` in `alphas`, compute the 5-fold
cross-validated MSE on the training set of `make_pipeline(PolynomialFeatures(15), StandardScaler(), Ridge(alpha))`
with `cross_val_score(..., scoring="neg_mean_squared_error")`. Set `cv_err` and `best_alpha`.

In [ ]:
alphas = [1e-6, 1e-4, 1e-2, 1, 10, 100]
cv_err = best_alpha = None
# YOUR CODE HERE (stuck? hint(3), then answer(3))
...
print(dict(zip(alphas, np.round(cv_err, 4))) if cv_err else None, " best:", best_alpha)

In [ ]:
check(3, lambda: cv_err[int(np.argmin(cv_err))] < cv_err[0] and best_alpha in (1e-2, 1, 10), 
      "Moderate regularization (alpha in [0.01, 1, 10]) should achieve lower CV error than tiny alpha (1e-6).")

## 4. Learning curves
`learning_curve` refits a model on growing slices of the data. For the degree-15 polynomial with
`Ridge(alpha=1e-6)` (barely regularized), get train and validation errors at 8 training sizes (5-fold CV over
all 300 points). Plot them: the gap between the curves is overfitting, and more data closes it.

In [ ]:
sizes = gap_small = gap_big = None
# YOUR CODE HERE (stuck? hint(4), then answer(4))
...
print(gap_small, gap_big)

In [ ]:
check(4, lambda: gap_big < gap_small / 5, 
      "With full dataset size, the generalization gap (validation error - training error) should shrink by > 5x.")

## 5. Leakage: fooling yourself on pure noise
100 examples, 5,000 features of random noise, random labels: nothing can be learned, so honest accuracy is ~50%.
The **leaky** recipe picks the 20 features most correlated with the labels *using all the data*, then
cross-validates. The **honest** recipe does the selection inside a pipeline, so each fold picks its features
from its own training part only. Set `leaky` and `honest` (mean 5-fold accuracy).

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif
Xn = rng.standard_normal((100, 5000)); yn = rng.integers(0, 2, 100)
leaky = honest = None
# YOUR CODE HERE (stuck? hint(5), then answer(5))
...
print("leaky:", leaky, " honest:", honest)

In [ ]:
check(5, lambda: leaky > 0.7 and honest < 0.65, 
      "Feature selection leakage produces artificially high CV accuracy (> 70%), while honest pipeline stays near chance (< 65%).")

Any step that looks at the data (scaling, feature selection, filling gaps, picking hyperparameters) must be fit
on training data only: put it in the pipeline. And the test set from part 1? Touch it once, at the very end.

In [ ]:
if Xtr is not None:
    final = make_pipeline(PolynomialFeatures(best_d or 5), LinearRegression()).fit(np.vstack([Xtr, Xva]), np.concatenate([ytr, yva]))
    print("test MSE, measured once:", np.mean((final.predict(Xte) - yte) ** 2))

## Next
Back to the path: https://ihor.sh/ai/ · next: chapter 12, measuring a model.